# Speech Emotion Recognition - WavLM v7 · Optimized for Base-Plus + Label Noise Handling

**Target emosi:** Netral · Senang · Sedih · Marah · Takut · Jijik

## Perubahan v6 → v7

| # | Masalah di v6 | Perbaikan v7 |
|---|---|---|
| ① | Dropout 0.50 terlalu berat utk base-plus | Turun ke 0.30 |
| ② | Full unfreeze 24 layer (rusak pretrained) | Kembali ke gradual unfreeze 6 layer |
| ③ | Mixup 50% probability terlalu agresif | Turun ke 25%, alpha 0.3→0.2 |
| ④ | SpecAugment terlalu berat | Probability 0.5→0.3, durasi 15%→8% |
| ⑤ | Label smoothing 0.15 | Turun ke 0.10 |
| ⑥ | **CREMA-D label noise tidak ditangani** | **Downweight per-kelas berdasar riset agreement rate** |
| ⑦ | Audio 6 detik (banyak silence) | Dipotong ke 4 detik |
| ⑧ | Tidak ada deteksi noise otomatis | Confidence-based noise detection + reweighting |

### Dasar ilmiah downweight CREMA-D
Penelitian CHUCKLE (2024, arXiv:2510.09382) menemukan agreement audio-only antara label intended vs perceived di CREMA-D sangat bervariasi per emosi:

| Emosi | Agreement Audio-only |
|---|---|
| Netral | 95.7% |
| Marah | 60.6% |
| Jijik | 30.0% |
| Takut | 32.0% |
| Senang | 26.0% |
| **Sedih** | **16.4%** [WARNING] |

Kelas Sedih di CREMA-D nyaris setara random noise untuk audio-only - diturunkan bobotnya signifikan.

> [WARNING] **Catatan penting:** Target 85% test accuracy untuk SER 6-kelas multi-corpus berada di luar rentang umum riset SOTA (biasanya 70-82%). Notebook ini menerapkan semua strategi yang masuk akal, tapi hasil aktual tergantung training run - tidak bisa dijamin sebelumnya.

## 1. Install & Import

**[1.1]** Menyiapkan environment (Kaggle/Colab/Lokal) dan instalasi pustaka-pustaka esensial MLOps seperti `transformers`, `accelerate`, dan `librosa`.


In [ ]:
import subprocess, sys, os
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules
ENV_NAME  = 'Kaggle' if IS_KAGGLE else ('Colab' if IS_COLAB else 'Lokal')
print(f'[ENV]  Lingkungan: {ENV_NAME}')
pkgs = ['transformers','accelerate','librosa','soundfile','tqdm','scikit-learn','seaborn']
subprocess.run([sys.executable,'-m','pip','install','-q',*pkgs], check=False)
print('[OK] Install selesai')

**[1.2]** Mengimpor semua modul Python yang akan digunakan untuk pengolahan data numerik, visualisasi, dan pemodelan PyTorch.


In [ ]:
import os, re, glob, json, random, warnings, math, time, shutil, zipfile
from contextlib import nullcontext
import numpy as np
import pandas as pd
import librosa, librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, get_cosine_schedule_with_warmup
from tqdm.notebook import tqdm
from IPython.display import FileLink, display as ipy_display
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('[WARNING]  GPU tidak terdeteksi.')

## 2. Konfigurasi Global - v7

Semua angka di sini sudah dikoreksi untuk `wavlm-base-plus` (bukan large). Lihat tabel perubahan di atas.

**[2.1]** Mendefinisikan *path* direktori global (input/output/checkpoint) dan parameter inti (seed, sampel rate 16kHz, batas durasi 4.0s).


In [ ]:
# ── PATH ──────────────────────────────────────────────────────────────
BASE_DIR      = '/kaggle/working' if IS_KAGGLE else os.getcwd()
DATA_DIR      = os.path.join(BASE_DIR, 'data')
RAW_DIR       = os.path.join(DATA_DIR, 'raw')
AUG_DIR       = os.path.join(DATA_DIR, 'augmented')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
MODEL_DIR     = os.path.join(BASE_DIR, 'models')
LOG_DIR       = os.path.join(BASE_DIR, 'logs')
for d in [RAW_DIR, AUG_DIR, PROCESSED_DIR, MODEL_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── VERIFIKASI & PATH DATASET KAGGLE / LOKAL ──────────────────────────
DATASET_CANDIDATES = {
    'IndoWaveSentiment': [
        '/kaggle/input/datasets/raditya1/indo-wave/IndoWaveSentiment',
        '/kaggle/input/indo-wave/IndoWaveSentiment',
        os.path.join(RAW_DIR, 'IndoWaveSentiment'),
        os.path.join(BASE_DIR, 'pipeline', 'Dataset', 'IndoWaveSentiment'),
    ],
    'RAVDESS': [
        '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio',
        '/kaggle/input/ravdess-emotional-speech-audio',
        os.path.join(RAW_DIR, 'ravdess'),
    ],
    'EmoDB': [
        '/kaggle/input/datasets/piyushagni5/berlin-database-of-emotional-speech-emodb/wav',
        '/kaggle/input/berlin-database-of-emotional-speech-emodb/wav',
        os.path.join(RAW_DIR, 'emodb'),
    ],
    'CREMA-D': [
        '/kaggle/input/datasets/ejlok1/cremad/AudioWAV',
        '/kaggle/input/cremad/AudioWAV',
        os.path.join(RAW_DIR, 'cremad'),
    ],
    'E-SERAVD': [
        '/kaggle/input/datasets/ahmadnafibudianto/e-seravd-speech-emotion-recognition-av-dataset/E-SERAVD 1.1/E-SERAVD/02_Audio-Only',
        '/kaggle/input/datasets/ahmadnafibudianto/e-seravd-speech-emotion-recognition-av-dataset/E-SERAVD/E-SERAVD/02_Audio-Only',
        '/kaggle/input/datasets/ahmadnafibudianto/e-seravd-speech-emotion-recognition-av-dataset/E-SERAVD 1.2/E-SERAVD 1.2/E-SERAVD/02_Audio-Only',
        os.path.join(RAW_DIR, 'eseravd'),
    ]
}

DATASET_PATHS = {}
found_count = 0

print('[CHECK] Memeriksa keberadaan dataset...')
for ds_name, candidates in DATASET_CANDIDATES.items():
    detected_path = next((p for p in candidates if os.path.isdir(p)), None)
    if detected_path:
        DATASET_PATHS[ds_name] = detected_path
        found_count += 1
        print(f'  [OK] {ds_name:<18}: Ditemukan -> {detected_path}')
    else:
        DATASET_PATHS[ds_name] = None
        print(f'  [NOT FOUND] {ds_name:<18}: TIDAK DITEMUKAN')

# Mapping ke variabel legacy agar cell berikutnya tetap kompatibel
KAGGLE_INDOWAVE = DATASET_PATHS['IndoWaveSentiment'] or '/kaggle/input/indowave-placeholder'
KAGGLE_RAVDESS  = DATASET_PATHS['RAVDESS'] or '/kaggle/input/ravdess-placeholder'
KAGGLE_EMODB    = DATASET_PATHS['EmoDB'] or '/kaggle/input/emodb-placeholder'
KAGGLE_CREMAD   = DATASET_PATHS['CREMA-D'] or '/kaggle/input/cremad-placeholder'
KAGGLE_ESERAVD  = DATASET_PATHS['E-SERAVD'] or '/kaggle/input/e-seravd-placeholder'

if found_count == 0:
    print('\n[ERROR] ERROR KRITIS: Tidak ada satu pun dataset yang terdeteksi di sistem!')
    print('   Sistem menghentikan eksekusi cell untuk mencegah proses training sia-sia.')
    sys.exit('Eksekusi dihentikan: Dataset tidak ditemukan.')

print(f'\n[NOTE] Total dataset terdeteksi: {found_count}/{len(DATASET_CANDIDATES)}')

# ── MODEL ─────────────────────────────────────────────────────────────
PRETRAINED_MODEL = 'microsoft/wavlm-base-plus'   # T4/P100 tidak cukup utk wavlm-large
LOCAL_PRETRAINED = None
MODEL_SOURCE = LOCAL_PRETRAINED if LOCAL_PRETRAINED and os.path.isdir(LOCAL_PRETRAINED) else PRETRAINED_MODEL

# ── AUDIO ─────────────────────────────────────────────────────────────
SAMPLE_RATE = 16000
MAX_SECONDS = 4.0          # ↓ dari 6.0 - emosi terkonsentrasi di awal ucapan
MIN_SECONDS = 0.35
MAX_SAMPLES = int(SAMPLE_RATE * MAX_SECONDS)
MIN_SAMPLES = int(SAMPLE_RATE * MIN_SECONDS)
AUDIO_EXTS  = ('.wav','.mp3','.flac','.ogg','.m4a','.mp4')

# ── LABEL ─────────────────────────────────────────────────────────────
EMOSI_LIST  = ['netral','senang','sedih','marah','takut','jijik']
LABEL2IDX   = {e:i for i,e in enumerate(EMOSI_LIST)}
IDX2LABEL   = {i:e for e,i in LABEL2IDX.items()}
NUM_CLASSES = len(EMOSI_LIST)
EMOJI_MAP   = {'netral':'😐','senang':'😊','sedih':'😢','marah':'😡','takut':'😨','jijik':'🤢'}

# ── BOBOT SUMBER DASAR ────────────────────────────────────────────────
BOBOT_SUMBER = {
    'indowavesentiment'    : 3.0,
    'indowavesentiment_aug': 2.0,
    'eseravd'              : 3.0,
    'eseravd_aug'          : 2.0,
    'ravdess'              : 0.7,
    'emodb'                : 0.7,
    'cremad'                : 0.7,   # dasar - akan di-modifikasi per-kelas di bawah
}

# ── BOBOT CREMA-D PER-KELAS (dari riset CHUCKLE 2024, audio-only agreement) ──
# Agreement rendah → bobot rendah (label kemungkinan noise)
CREMAD_AGREEMENT = {
    'netral' : 0.957,   # 95.7% agreement → bobot tinggi, dipertahankan
    'marah'  : 0.606,   # 60.6%
    'jijik'  : 0.300,   # 30.0%
    'takut'  : 0.320,   # 32.0%
    'senang' : 0.260,   # 26.0%
    'sedih'  : 0.164,   # 16.4% ← hampir random, downweight ekstrem
}
# Skala bobot: clip minimum 0.10 supaya tidak hilang total (masih ada sinyal)
CREMAD_BOBOT_PER_KELAS = {
    e: max(0.10, agree * 1.0) for e, agree in CREMAD_AGREEMENT.items()
}
print('Bobot CREMA-D per kelas (dari agreement rate):')
for e, w in CREMAD_BOBOT_PER_KELAS.items():
    print(f'   {EMOJI_MAP[e]} {e:<8}: {w:.3f}')

# ── SPLIT ─────────────────────────────────────────────────────────────
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15
SEED       = 42

# ── AUGMENTASI OFFLINE ────────────────────────────────────────────────
DO_AUGMENT = True
AUG_MODES  = ['pitch_up2','pitch_down2','stretch_slow','stretch_fast','noise_light']
# pitch_up1 dihapus - terlalu mirip pitch_up2, mengurangi diversity riil

# ── HYPERPARAMETER v7 (dikoreksi utk base-plus) ───────────────────────
BATCH_SIZE        = 6
GRAD_ACCUM_STEPS  = 2
EPOCHS            = 22
FREEZE_EPOCHS     = 2
UNFREEZE_LAST_N   = 6          # ← balik ke gradual (v5 style), bukan full unfreeze
BACKBONE_LR       = 1e-5       # ← naik dikit dari v6 (8e-6), krn cuma 6 layer
HEAD_LR           = 3e-4
WEIGHT_DECAY      = 1e-2
WARMUP_RATIO      = 0.10
EARLY_STOP        = 6
MIN_DELTA         = 0.001
GRAD_CLIP         = 1.0
DROPOUT           = 0.30       # ↓ dari 0.50
LABEL_SMOOTHING   = 0.10       # ↓ dari 0.15
USE_MIXUP         = True
MIXUP_ALPHA       = 0.2        # ↓ dari 0.3
MIXUP_PROB        = 0.25       # ↓ dari implisit 0.5
USE_SPECAUGMENT   = True
SPEC_TIME_MASK_P  = 0.3        # ↓ dari 0.5
SPEC_TIME_MAX     = int(0.08 * MAX_SAMPLES)   # ↓ dari 0.15
USE_AMP           = True
USE_GRAD_CKPT     = True
NUM_WORKERS       = 2 if IS_KAGGLE else 0

# ── LABEL NOISE DETECTION (otomatis setelah training awal) ────────────
DO_NOISE_DETECTION = True
NOISE_CONF_THRESHOLD = 0.40    # confidence di bawah ini + salah prediksi = kandidat noise
NOISE_DOWNWEIGHT = 0.3         # bobot file yang dicurigai noise

BEST_MODEL_PATH      = os.path.join(MODEL_DIR, 'ser_wavlm_v7_best.pt')
BEST_MODEL_PATH_STG2 = os.path.join(MODEL_DIR, 'ser_wavlm_v7_stage2_best.pt')
HISTORY_PATH    = os.path.join(LOG_DIR,   'history_v7.json')
SPLIT_PATH      = os.path.join(DATA_DIR,  'metadata_split_v7.csv')

print(f'\n[OK] Konfigurasi v7 siap')
print(f'   Model         : {MODEL_SOURCE}')
print(f'   Audio durasi  : {MAX_SECONDS}s (dipotong dari 6.0s)')
print(f'   Dropout       : {DROPOUT}')
print(f'   Unfreeze      : {UNFREEZE_LAST_N} layer (gradual)')
print(f'   Mixup         : prob={MIXUP_PROB}, alpha={MIXUP_ALPHA}')
print(f'   SpecAugment   : prob={SPEC_TIME_MASK_P}, max_dur=8%')
print(f'   Label smooth  : {LABEL_SMOOTHING}')
print(f'   Noise detect  : {DO_NOISE_DETECTION}')
print(f'   Device        : {device}')

## 3. Panduan Dataset & Parser

**[3.1]** Mendefinisikan *dictionary mapping* untuk menstandardisasi label kelas numerik multi-dataset (IndoWave, RAVDESS, EmoDB) ke dalam standar label emosi Bahasa Indonesia.


In [ ]:
INDOWAVE_MAP = {'01':'netral','02':'senang','03':None,'04':'jijik','05':'sedih'}
RAVDESS_MAP  = {'01':'netral','02':'netral','03':'senang','04':'sedih',
                '05':'marah','06':'takut','07':'jijik','08':None}
EMODB_MAP    = {'W':'marah','L':None,'E':'jijik','A':'takut','F':'senang','T':'sedih','N':'netral'}
CREMAD_MAP   = {'ANG':'marah','DIS':'jijik','FEA':'takut','HAP':'senang','NEU':'netral','SAD':'sedih'}
ESERAVD_MAP  = {'angry':'marah','sad':'sedih','neutral':'netral',
                'happy':'senang','fear':'takut','disgust':'jijik'}

def _buat_baris(fp, emosi, sumber, bahasa):
    return {'path':fp,'emosi':emosi,'label':LABEL2IDX[emosi],
            'sumber':sumber,'bahasa':bahasa,'file':os.path.basename(fp)}

def _is_audio(fp):
    return os.path.isfile(fp) and os.path.splitext(fp)[1].lower() in AUDIO_EXTS

def parse_indowavesentiment(base_dir):
    rows=[]
    for fp in glob.glob(os.path.join(base_dir,'**','*'),recursive=True):
        if not _is_audio(fp): continue
        parts=os.path.splitext(os.path.basename(fp))[0].split('-')
        if len(parts)<2: continue
        emosi=INDOWAVE_MAP.get(parts[1].zfill(2))
        if emosi and emosi in LABEL2IDX: rows.append(_buat_baris(fp,emosi,'indowavesentiment','id'))
    df=pd.DataFrame(rows); print(f'  [OK] IndoWaveSentiment : {len(df):>4} file'); return df

def parse_ravdess(base_dir):
    rows=[]
    for fp in glob.glob(os.path.join(base_dir,'**','*'),recursive=True):
        if not _is_audio(fp): continue
        parts=os.path.splitext(os.path.basename(fp))[0].split('-')
        if len(parts)<7: continue
        emosi=RAVDESS_MAP.get(parts[2])
        if emosi and emosi in LABEL2IDX: rows.append(_buat_baris(fp,emosi,'ravdess','en'))
    df=pd.DataFrame(rows); print(f'  [OK] RAVDESS           : {len(df):>4} file'); return df

def parse_emodb(base_dir):
    rows=[]
    for fp in glob.glob(os.path.join(base_dir,'**','*'),recursive=True):
        if not _is_audio(fp): continue
        nama=os.path.splitext(os.path.basename(fp))[0]
        if len(nama)<6: continue
        emosi=EMODB_MAP.get(nama[5].upper())
        if emosi and emosi in LABEL2IDX: rows.append(_buat_baris(fp,emosi,'emodb','de'))
    df=pd.DataFrame(rows); print(f'  [OK] EmoDB (Berlin)    : {len(df):>4} file'); return df

def parse_cremad(base_dir):
    rows=[]
    for fp in glob.glob(os.path.join(base_dir,'**','*'),recursive=True):
        if not _is_audio(fp): continue
        parts=os.path.splitext(os.path.basename(fp))[0].split('_')
        if len(parts)<3: continue
        emosi=CREMAD_MAP.get(parts[2].upper())
        if emosi and emosi in LABEL2IDX: rows.append(_buat_baris(fp,emosi,'cremad','en'))
    df=pd.DataFrame(rows); print(f'  [OK] CREMA-D           : {len(df):>4} file'); return df

def normalisasi_label_eseravd(nama_folder):
    nama = nama_folder.lower().strip()
    return re.sub(r'^\d+[_\-\s]*', '', nama)

def ambil_label_eseravd_dari_path(fp, base_dir):
    rel_path = os.path.relpath(fp, base_dir)
    for part in rel_path.split(os.sep):
        label_folder = normalisasi_label_eseravd(part)
        if label_folder in ESERAVD_MAP: return ESERAVD_MAP[label_folder]
    return None

def parse_eseravd(base_dir):
    rows=[]; total_audio=0; gagal_label=0
    for fp in glob.glob(os.path.join(base_dir,'**','*'),recursive=True):
        if not _is_audio(fp): continue
        total_audio += 1
        emosi = ambil_label_eseravd_dari_path(fp, base_dir)
        if emosi and emosi in LABEL2IDX: rows.append(_buat_baris(fp,emosi,'eseravd','id'))
        else: gagal_label += 1
    df=pd.DataFrame(rows)
    print(f'  [OK] E-SERAVD          : {len(df):>4} file (gagal label: {gagal_label})')
    return df

print('[OK] Fungsi parser siap')

**[3.2]** Memuat (*load*) metadata seluruh dataset, mengeleminasi path audio yang rusak (missing/corrupt), dan menggabungkannya ke dalam satu DataFrame raksasa.


In [ ]:
print('[LOAD] Memuat semua dataset...')
dfs = []
for fn, path in [
    (parse_indowavesentiment, KAGGLE_INDOWAVE),
    (parse_ravdess,           KAGGLE_RAVDESS),
    (parse_emodb,             KAGGLE_EMODB),
    (parse_cremad,            KAGGLE_CREMAD),
    (parse_eseravd,           KAGGLE_ESERAVD),
]:
    if os.path.isdir(path):
        tmp = fn(path)
        if len(tmp): dfs.append(tmp)
    else:
        print(f'  [WARNING]  Tidak ditemukan: {path}')

if not dfs:
    raise RuntimeError('Tidak ada dataset yang berhasil dimuat.')

df = pd.concat(dfs, ignore_index=True)
df = df.drop_duplicates(subset=['path']).reset_index(drop=True)

# ── Hitung bobot dengan koreksi khusus CREMA-D per-kelas ──────────────
def hitung_bobot(row):
    bobot_dasar = BOBOT_SUMBER.get(row['sumber'], 1.0)
    if row['sumber'] == 'cremad':
        return bobot_dasar * CREMAD_BOBOT_PER_KELAS.get(row['emosi'], 0.5)
    return bobot_dasar

df['bobot'] = df.apply(hitung_bobot, axis=1)
df.to_csv(os.path.join(DATA_DIR,'metadata_gabungan_v7.csv'), index=False)

print(f'\n[STATS] Total file unik (sebelum augmentasi): {len(df)}')
print('\nDistribusi sumber:')
print(df.groupby('sumber')[['emosi']].count().rename(columns={'emosi':'total'}))
print('\nDistribusi emosi:')
print(df.groupby('emosi').size().reindex(EMOSI_LIST).fillna(0).astype(int).rename('total'))
print('\nBobot rata-rata CREMA-D per emosi (sudah dikoreksi):')
print(df[df['sumber']=='cremad'].groupby('emosi')['bobot'].mean().round(3))
n_indo = (df['bahasa']=='id').sum()
print(f'\n[NOTE] Proporsi data Indonesia: {n_indo}/{len(df)} ({n_indo/len(df)*100:.1f}%)')

## 4. Augmentasi Offline - Data Indonesia

Sama seperti v5/v6, tapi `pitch_up1` dihapus karena terlalu mirip `pitch_up2` (mengurangi diversity riil, hanya menambah jumlah file).

**[4.1]** Mendefinisikan fungsi augmentasi *offline* khusus (Time Stretch, Pitch Shift) untuk memanipulasi dan menggandakan kuantitas data latih Indonesia.


In [ ]:
def load_for_aug(path, sr=SAMPLE_RATE):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        y = np.nan_to_num(y.astype(np.float32))
        peak = np.max(np.abs(y))
        if peak > 1e-5: y = y / peak
        return y
    except: return None

def augment_offline(y, sr, mode):
    if mode == 'pitch_up2':   return librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
    elif mode == 'pitch_down2': return librosa.effects.pitch_shift(y, sr=sr, n_steps=-2)
    elif mode == 'stretch_slow':
        y_aug = librosa.effects.time_stretch(y, rate=0.85)
        if len(y_aug)>len(y): y_aug=y_aug[:len(y)]
        elif len(y_aug)<len(y): y_aug=np.pad(y_aug,(0,len(y)-len(y_aug)))
        return y_aug
    elif mode == 'stretch_fast':
        y_aug = librosa.effects.time_stretch(y, rate=1.15)
        if len(y_aug)>len(y): y_aug=y_aug[:len(y)]
        elif len(y_aug)<len(y): y_aug=np.pad(y_aug,(0,len(y)-len(y_aug)))
        return y_aug
    elif mode == 'noise_light':
        return y + 0.004 * np.random.randn(len(y)).astype(np.float32)
    return y

def run_augmentasi_offline(df_src, sumber_tag, aug_dir, aug_modes):
    rows_aug = []
    df_target = df_src[df_src['sumber']==sumber_tag].copy()
    print(f'   Augmenting {sumber_tag}: {len(df_target)} file × {len(aug_modes)} mode = {len(df_target)*len(aug_modes)} file baru')
    for _, row in tqdm(df_target.iterrows(), total=len(df_target), desc=f'Aug {sumber_tag}'):
        y = load_for_aug(row['path'])
        if y is None or len(y) < MIN_SAMPLES: continue
        for mode in aug_modes:
            try:
                y_aug = np.clip(augment_offline(y, SAMPLE_RATE, mode), -1.0, 1.0).astype(np.float32)
                fname = f"aug_{mode}_{os.path.splitext(os.path.basename(row['path']))[0]}.wav"
                out_path = os.path.join(aug_dir, sumber_tag, fname)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                sf.write(out_path, y_aug, SAMPLE_RATE)
                rows_aug.append({'path':out_path,'emosi':row['emosi'],'label':row['label'],
                    'sumber':f'{sumber_tag}_aug','bahasa':row['bahasa'],'file':fname,
                    'bobot':BOBOT_SUMBER.get(f'{sumber_tag}_aug',2.0)})
            except: pass
    return pd.DataFrame(rows_aug)

df_aug_parts = []
if DO_AUGMENT:
    print('[AUGMENT] Memulai augmentasi offline data Indonesia...')
    for tag in ['indowavesentiment','eseravd']:
        if tag in df['sumber'].values:
            tmp = run_augmentasi_offline(df, tag, AUG_DIR, AUG_MODES)
            if len(tmp): df_aug_parts.append(tmp)
    print('[OK] Augmentasi selesai.')
else:
    print('[LOAD] DO_AUGMENT=False - memuat dari disk...')
    for tag in ['indowavesentiment','eseravd']:
        for fp in glob.glob(os.path.join(AUG_DIR, tag, '*.wav')):
            orig = os.path.basename(fp)
            for mode in AUG_MODES: orig = orig.replace(f'aug_{mode}_','')
            orig = orig.replace('.wav','')
            match = df[df['file'].str.startswith(orig)]
            if len(match):
                row = match.iloc[0]
                df_aug_parts.append(pd.DataFrame([{'path':fp,'emosi':row['emosi'],'label':row['label'],
                    'sumber':f'{tag}_aug','bahasa':row['bahasa'],'file':os.path.basename(fp),
                    'bobot':BOBOT_SUMBER.get(f'{tag}_aug',2.0)}]))

if df_aug_parts:
    df_aug_all = pd.concat(df_aug_parts, ignore_index=True)
    df = pd.concat([df, df_aug_all], ignore_index=True).reset_index(drop=True)
    print(f'\n[STATS] Total setelah augmentasi: {len(df)} file')
    print(df.groupby('sumber').size().rename('total').sort_values(ascending=False))
else:
    print('[WARNING]  Tidak ada augmentasi.')

## 5. Split Data

**[5.1]** Memecah dataset terpadu menjadi partisi *Train*, *Validation*, dan *Test Split* dengan stratifikasi kelas agar distribusi labelnya seimbang.


In [ ]:
df_orig = df[~df['sumber'].str.endswith('_aug')].copy()
df_augmented = df[df['sumber'].str.endswith('_aug')].copy()

df_trainval_orig, df_test = train_test_split(
    df_orig, test_size=TEST_SPLIT, stratify=df_orig['label'], random_state=SEED)
rv = VAL_SPLIT / (1 - TEST_SPLIT)
df_train_orig, df_val = train_test_split(
    df_trainval_orig, test_size=rv, stratify=df_trainval_orig['label'], random_state=SEED)

df_train = pd.concat([df_train_orig, df_augmented], ignore_index=True)
df_train = df_train.reset_index(drop=True)
df_train['split']='train'; df_val['split']='val'; df_test['split']='test'
df_split = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_split.to_csv(SPLIT_PATH, index=False)

print(f'Split → Latih: {len(df_train)} | Val: {len(df_val)} | Uji: {len(df_test)}')
print(f'   (termasuk {len(df_augmented)} file augmentasi di train)')
print('\nNote: val/test TETAP pakai semua data asli CREMA-D apa adanya')
print('(termasuk yang label-nya mungkin noisy) - supaya evaluasi tetap realistis')
print('terhadap data dunia nyata, bukan di-cherry-pick.')

## 6. Audio Loading + SpecAugment (versi ringan)

**[6.1]** Fungsi pemuatan sinyal *waveform* audio, pemotongan keheningan (*silence trimming* 30dB), normalisasi amplitudo puncak, dan mekanisme *padding/crop* durasi.


In [ ]:
def load_waveform(path, sr=SAMPLE_RATE):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        if y is None or len(y)==0: return np.zeros(MIN_SAMPLES, dtype=np.float32)
        y = np.nan_to_num(y.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        yt, _ = librosa.effects.trim(y, top_db=30)
        if len(yt)>=MIN_SAMPLES: y=yt
        peak = np.max(np.abs(y)) if len(y) else 0.0
        if peak>1e-5: y=y/peak
        return y.astype(np.float32)
    except: return np.zeros(MIN_SAMPLES, dtype=np.float32)

def fix_length(y, mode='train'):
    if len(y)>MAX_SAMPLES:
        start = np.random.randint(0,len(y)-MAX_SAMPLES+1) if mode=='train' else max(0,(len(y)-MAX_SAMPLES)//2)
        y = y[start:start+MAX_SAMPLES]
    elif len(y)<MAX_SAMPLES:
        y = np.pad(y,(0,MAX_SAMPLES-len(y)),mode='constant')
    return y.astype(np.float32)

def spec_augment(y):
    if np.random.rand() >= SPEC_TIME_MASK_P: return y
    y = y.copy()
    mask_len = np.random.randint(int(0.02*SAMPLE_RATE), max(int(0.02*SAMPLE_RATE)+1, SPEC_TIME_MAX))
    mask_start = np.random.randint(0, max(1, len(y)-mask_len))
    y[mask_start:mask_start+mask_len] = 0.0
    return y

def augment_light(y):
    if np.random.rand()<0.40: y = y * np.random.uniform(0.80, 1.20)
    if np.random.rand()<0.30: y = y + np.random.randn(len(y)).astype(np.float32)*np.random.uniform(0.001,0.004)
    if np.random.rand()<0.25:
        shift = int(np.random.uniform(-0.08,0.08)*SAMPLE_RATE)
        y = np.roll(y, shift)
    if USE_SPECAUGMENT: y = spec_augment(y)
    return np.clip(y, -1.0, 1.0).astype(np.float32)

class SERWaveformDataset(Dataset):
    def __init__(self, frame, mode='train'):
        self.df = frame.reset_index(drop=True).copy()
        self.mode = mode
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = load_waveform(row['path'])
        y = fix_length(y, mode=self.mode)
        if self.mode=='train': y = augment_light(y)
        meta = {'path':row['path'],'emosi':row['emosi'],'sumber':row['sumber'],'bahasa':row['bahasa']}
        return y, int(row['label']), meta

print('[OK] Audio loader siap (durasi 4s, SpecAugment ringan)')

## 7. DataLoader & Mixup (intensitas dikurangi)

**[7.1]** Menginisialisasi `AutoFeatureExtractor` Hugging Face dan mendefinisikan `collate_fn` untuk perataan (*padding*) dinamis tensor audio ke DataLoader.


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_SOURCE)
print('[OK] Feature extractor dimuat:', MODEL_SOURCE)

def collate_fn(batch):
    waves, labels, metas = zip(*batch)
    inputs = feature_extractor(
        list(waves), sampling_rate=SAMPLE_RATE, padding=True,
        truncation=True, max_length=MAX_SAMPLES,
        return_attention_mask=True, return_tensors='pt'
    )
    if 'attention_mask' not in inputs:
        inputs['attention_mask'] = torch.ones_like(inputs['input_values'], dtype=torch.long)
    return inputs, torch.tensor(labels, dtype=torch.long), metas

def mixup_batch(inputs, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return inputs, F.one_hot(labels, NUM_CLASSES).float()
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)
    idx = torch.randperm(labels.size(0), device=labels.device)
    inputs['input_values'] = lam * inputs['input_values'] + (1-lam) * inputs['input_values'][idx]
    if 'attention_mask' in inputs:
        inputs['attention_mask'] = torch.maximum(inputs['attention_mask'], inputs['attention_mask'][idx])
    labels_mixed = lam * F.one_hot(labels, NUM_CLASSES).float() + (1-lam) * F.one_hot(labels[idx], NUM_CLASSES).float()
    return inputs, labels_mixed

classes      = np.arange(NUM_CLASSES)
y_train_np   = df_train['label'].values.astype(int)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train_np)
class_weights = np.clip(class_weights, 0.60, 2.50).astype(np.float32)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)
print('Bobot kelas:', dict(zip(EMOSI_LIST, np.round(class_weights, 2))))

source_w = df_train['bobot'].values.astype(np.float32)
sample_w = class_weights[y_train_np] * np.sqrt(np.clip(source_w, 0.1, 3.0))
lo, hi   = np.percentile(sample_w, [2, 98])
sample_w = np.clip(sample_w, lo, hi)
sampler  = WeightedRandomSampler(weights=torch.DoubleTensor(sample_w),
               num_samples=len(sample_w), replacement=True)

train_ds = SERWaveformDataset(df_train, mode='train')
val_ds   = SERWaveformDataset(df_val,   mode='eval')
test_ds  = SERWaveformDataset(df_test,  mode='eval')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)

print(f'[OK] DataLoader siap | train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}')

## 8. Arsitektur Model - WavLM-Base-Plus + Attentive Stats Pooling

**[8.1]** Membangun arsitektur PyTorch: *backbone* `WavLM-Base-Plus` disambung dengan *custom head* berupa layer `AttentiveStatsPooling` dan linear *classifier*.


In [ ]:
class AttentiveStatsPooling(nn.Module):
    def __init__(self, hidden_size, attn_hidden=128, dropout=0.2):
        super().__init__()
        self.attn = nn.Sequential(
            nn.LayerNorm(hidden_size), nn.Linear(hidden_size, attn_hidden),
            nn.Tanh(), nn.Dropout(dropout), nn.Linear(attn_hidden, 1)
        )
    def forward(self, x, mask=None):
        score = self.attn(x).squeeze(-1)
        if mask is not None: score = score.masked_fill(~mask.bool(), -1e4)
        weight = torch.softmax(score, dim=-1).unsqueeze(-1)
        mean = torch.sum(weight * x, dim=1)
        var  = torch.sum(weight * (x - mean.unsqueeze(1))**2, dim=1)
        std  = torch.sqrt(var.clamp(min=1e-6))
        return torch.cat([mean, std], dim=-1), weight.squeeze(-1)

class WavLMSER(nn.Module):
    def __init__(self, model_source, num_labels, dropout=0.30):
        super().__init__()
        self.config   = AutoConfig.from_pretrained(model_source)
        self.backbone = AutoModel.from_pretrained(model_source)
        hidden = self.config.hidden_size
        self.pooling    = AttentiveStatsPooling(hidden, attn_hidden=128, dropout=dropout*0.5)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden*2), nn.Dropout(dropout),
            nn.Linear(hidden*2, 256), nn.GELU(),
            nn.Dropout(dropout*0.75), nn.Linear(256, num_labels)
        )
    def _feature_mask(self, hidden, attention_mask):
        if attention_mask is None: return None
        if hasattr(self.backbone, '_get_feature_vector_attention_mask'):
            return self.backbone._get_feature_vector_attention_mask(hidden.shape[1], attention_mask)
        return None
    def forward(self, input_values, attention_mask=None):
        out    = self.backbone(input_values=input_values, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        pooled, attn = self.pooling(hidden, self._feature_mask(hidden, attention_mask))
        return self.classifier(pooled), attn

def _get_core(m):
    return m.module if hasattr(m, 'module') else m

def freeze_all_backbone(model):
    core = _get_core(model)
    for p in core.backbone.parameters(): p.requires_grad = False
    for p in core.pooling.parameters():  p.requires_grad = True
    for p in core.classifier.parameters(): p.requires_grad = True

def unfreeze_last_layers(model, last_n=6):
    for p in core.backbone.parameters(): p.requires_grad = False
    layers = getattr(core.backbone.encoder, 'layers', getattr(core.backbone.encoder, 'layer', None))
    if layers is not None:
        for layer in layers[-last_n:]:
            for p in layer.parameters(): p.requires_grad = True
    for p in core.pooling.parameters():    p.requires_grad = True
    for p in core.classifier.parameters(): p.requires_grad = True

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = WavLMSER(MODEL_SOURCE, NUM_CLASSES, dropout=DROPOUT).to(device)
if USE_GRAD_CKPT and hasattr(model.backbone, 'gradient_checkpointing_enable'):
    model.backbone.gradient_checkpointing_enable()
    print('✅ Gradient checkpointing aktif')

if torch.cuda.device_count() > 1:
    print(f'✅ Multi-GPU terdeteksi! Menggunakan {torch.cuda.device_count()} GPU (DataParallel)')
    model = nn.DataParallel(model)

total, trainable = count_params(model)
print(f'[OK] Model WavLMSER siap | total: {total:,} | trainable: {trainable:,}')

## 9. Loss, Optimizer, Scheduler

**[9.1]** Menginisialisasi fungsi objektif (Cross Entropy Loss dengan Class Weights), logika augmentasi *Mixup*, Optimizer AdamW, dan Learning Rate Scheduler kosinus.


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTHING)

def mixup_loss(logits, labels_soft):
    log_probs = F.log_softmax(logits, dim=-1)
    return -(labels_soft * log_probs).sum(dim=-1).mean()

optimizer = torch.optim.AdamW([
    {'params': _get_core(model).backbone.parameters(),   'lr': BACKBONE_LR},
    {'params': _get_core(model).pooling.parameters(),    'lr': HEAD_LR},
    {'params': _get_core(model).classifier.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)

total_update_steps = math.ceil(len(train_loader)/GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps       = int(WARMUP_RATIO * total_update_steps)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_update_steps)

amp_enabled = bool(device.type=='cuda' and USE_AMP)
scaler      = torch.cuda.amp.GradScaler(enabled=amp_enabled)

print('[OK] Loss/optimizer/scheduler siap')
print(f'   Loss      : CrossEntropy + LabelSmooth({LABEL_SMOOTHING})')
print(f'   Mixup     : prob={MIXUP_PROB} alpha={MIXUP_ALPHA}')
print(f'   Warmup    : {warmup_steps}/{total_update_steps}')

## 10. Training Loop (Stage 1 - Semua Data)

**[10.1]** Fungsi-fungsi utilitas kalkulasi komputasi: ekstraksi akurasi metrik tensor, injeksi data ke VRAM GPU, dan mode komputasi *Mixed Precision* (AMP).


In [ ]:
def akurasi(logits, labels):
    return (logits.argmax(1)==labels).float().mean().item()
def to_device_inputs(inputs):
    return {k:v.to(device, non_blocking=True) for k,v in inputs.items()}
def autocast_ctx():
    return torch.cuda.amp.autocast() if amp_enabled else nullcontext()
def apply_trainable_schedule(epoch):
    if epoch <= FREEZE_EPOCHS:
        freeze_all_backbone(model); return 'freeze backbone (head only)'
    else:
        unfreeze_last_layers(model, last_n=UNFREEZE_LAST_N)
        return f'unfreeze last {UNFREEZE_LAST_N} layers'

def train_epoch(epoch):
    model.train()
    total_loss, total_acc, total_n = 0.0, 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} train', leave=False)
    for step, (inputs, labels, metas) in enumerate(pbar, start=1):
        inputs = to_device_inputs(inputs)
        labels = labels.to(device, non_blocking=True)
        use_mixup = USE_MIXUP and np.random.rand() < MIXUP_PROB
        if use_mixup:
            inputs, labels_soft = mixup_batch(inputs, labels, alpha=MIXUP_ALPHA)
        with autocast_ctx():
            logits, _ = model(**inputs)
            if use_mixup:
                loss = mixup_loss(logits, labels_soft) / GRAD_ACCUM_STEPS
            else:
                loss = criterion(logits, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        bs = labels.size(0)
        total_loss += loss.item() * GRAD_ACCUM_STEPS * bs
        total_acc  += akurasi(logits.detach(), labels) * bs
        total_n    += bs
        pbar.set_postfix(loss=total_loss/max(1,total_n), acc=total_acc/max(1,total_n))
    return total_loss/total_n, total_acc/total_n

@torch.no_grad()
def eval_epoch(loader, desc='eval'):
    model.eval()
    total_loss, total_acc, total_n = 0.0, 0.0, 0
    preds, trues, sumber_list, bahasa_list, paths = [], [], [], [], []
    for inputs, labels, metas in tqdm(loader, desc=desc, leave=False):
        inputs = to_device_inputs(inputs)
        labels = labels.to(device, non_blocking=True)
        with autocast_ctx():
            logits, _ = model(**inputs)
            loss = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item()*bs; total_acc += akurasi(logits,labels)*bs; total_n += bs
        preds.extend(logits.argmax(1).cpu().numpy().tolist())
        trues.extend(labels.cpu().numpy().tolist())
        sumber_list.extend([m['sumber'] for m in metas])
        bahasa_list.extend([m['bahasa'] for m in metas])
        paths.extend([m['path'] for m in metas])
    return {'loss':total_loss/total_n,'acc':total_acc/total_n,
            'pred':np.array(preds),'true':np.array(trues),
            'sumber':np.array(sumber_list),'bahasa':np.array(bahasa_list),'path':np.array(paths)}

print('[OK] Fungsi training/evaluasi siap')

**[10.2]** Menjalankan siklus iterasi (Epoch) pelatihan Stage 1: Melatih parameter model, memonitor degradasi loss, dan otomatis menyimpan checkpoint bobot terbaik.


In [ ]:
import os
riwayat = {'ll':[],'la':[],'vl':[],'va':[],'lr_backbone':[],'lr_head':[],'stage':[]}
best_acc, best_loss, best_epoch = 0.0, float('inf'), 0
best_train_acc, no_imp, last_stage = 0.0, 0, None
start_epoch = 1

if os.path.exists(BEST_MODEL_PATH):
    print(f'[INFO] Ditemukan checkpoint di {BEST_MODEL_PATH}. Memuat state...')
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    
    best_acc = ckpt.get('val_acc', 0.0)
    best_loss = ckpt.get('val_loss', float('inf'))
    best_epoch = ckpt.get('epoch', 0)
    best_train_acc = ckpt.get('train_acc', 0.0)
    
    if 'history' in ckpt:
        riwayat = ckpt['history']
        start_epoch = len(riwayat['la']) + 1
        no_imp = (start_epoch - 1) - best_epoch
        print(f'       Melanjutkan dari epoch {start_epoch}')
    if 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])

print(f'{"Epoch":>5} | {"Stage":<30} | {"Loss T":>8} | {"Acc T":>7} | {"Loss V":>8} | {"Acc V":>7} | {"Gap":>6}')
print('─' * 96)

for epoch in range(start_epoch, EPOCHS+1):
    stage = apply_trainable_schedule(epoch)
    if stage != last_stage:
        total_p, train_p = count_params(model)
        print(f'\n[CONFIG] Epoch {epoch}: {stage} | trainable {train_p:,}/{total_p:,} ({train_p/total_p*100:.1f}%)')
        last_stage = stage

    train_loss, train_acc = train_epoch(epoch)
    val_out   = eval_epoch(val_loader, desc=f'Ep{epoch} val')
    val_loss, val_acc = val_out['loss'], val_out['acc']

    lr_bb, lr_hd = optimizer.param_groups[0]['lr'], optimizer.param_groups[1]['lr']
    gap = train_acc - val_acc

    for k, v in zip(['ll','la','vl','va','lr_backbone','lr_head','stage'],
                    [train_loss,train_acc,val_loss,val_acc,lr_bb,lr_hd,stage]):
        riwayat[k].append(float(v) if k!='stage' else v)

    improved = (val_acc > best_acc + MIN_DELTA) or (abs(val_acc-best_acc)<=MIN_DELTA and val_loss<best_loss)
    flag = ''
    if improved:
        best_acc, best_loss, best_epoch, best_train_acc, no_imp = \
            float(val_acc), float(val_loss), int(epoch), float(train_acc), 0
        flag = ' [BEST]'
        torch.save({
            'epoch':best_epoch,'model_state':_get_core(model).state_dict(),
            'optimizer_state':optimizer.state_dict(),
            'model_source':MODEL_SOURCE,'pretrained_model':PRETRAINED_MODEL,
            'val_acc':best_acc,'val_loss':best_loss,
            'train_acc':best_train_acc,'train_loss':float(train_loss),
            'emosi_list':EMOSI_LIST,'label2idx':LABEL2IDX,'idx2label':IDX2LABEL,
            'class_weights':class_weights.tolist(),'history':riwayat,
            'cfg':{'sample_rate':SAMPLE_RATE,'max_seconds':MAX_SECONDS,'max_samples':MAX_SAMPLES,
                   'dropout':DROPOUT,'label_smoothing':LABEL_SMOOTHING,
                   'use_mixup':USE_MIXUP,'mixup_alpha':MIXUP_ALPHA,'mixup_prob':MIXUP_PROB,
                   'freeze_epochs':FREEZE_EPOCHS,'unfreeze_last_n':UNFREEZE_LAST_N,
                   'backbone_lr':BACKBONE_LR,'head_lr':HEAD_LR,
                   'batch_size':BATCH_SIZE,'grad_accum_steps':GRAD_ACCUM_STEPS,'seed':SEED}
        }, BEST_MODEL_PATH)
    else:
        no_imp += 1

    print(f'{epoch:>5} | {stage:<30} | {train_loss:>8.4f} | {train_acc:>7.4f} | {val_loss:>8.4f} | {val_acc:>7.4f} | {gap:>6.3f}{flag}')
    with open(HISTORY_PATH,'w',encoding='utf-8') as f: json.dump(riwayat,f,indent=2)
    if no_imp >= EARLY_STOP:
        print(f'\n[STOP] Early stop epoch {epoch} | best val acc: {best_acc:.4f} @ epoch {best_epoch}')
        break

ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state']); model.eval()
print(f'\n[OK] Stage 1 selesai | Best epoch: {best_epoch} | Best val acc: {best_acc*100:.2f}%')


## 11. Deteksi Label Noise Otomatis (Confidence-Based)

Setelah training awal, deteksi file di training set yang **diprediksi salah dengan confidence rendah** - kandidat label noise. Lalu turunkan bobotnya dan training ulang (Stage 2) singkat untuk fine-tune dengan data yang sudah dibersihkan.

**[11.1]** Inferensi pasca-pelatihan awal untuk mendeteksi *Label Noise*: Melacak sampel data latih yang salah diprediksi namun model memiliki *confidence score* ekstrem.


In [ ]:
@torch.no_grad()
def deteksi_label_noise(loader, threshold_conf=NOISE_CONF_THRESHOLD):
    model.eval()
    suspicious = []
    for inputs, labels, metas in tqdm(loader, desc='Deteksi noise', leave=False):
        inputs = to_device_inputs(inputs)
        labels = labels.to(device)
        logits, _ = model(**inputs)
        probs = F.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)
        for i in range(len(labels)):
            if pred[i] != labels[i] and conf[i] < threshold_conf:
                suspicious.append({
                    'path': metas[i]['path'], 'sumber': metas[i]['sumber'],
                    'label_asli': IDX2LABEL[labels[i].item()],
                    'prediksi': IDX2LABEL[pred[i].item()],
                    'confidence': conf[i].item()
                })
    return pd.DataFrame(suspicious)

if DO_NOISE_DETECTION:
    # Pakai loader train tanpa shuffle utk deteksi (pakai val-mode dataset supaya tidak ada augmentasi acak)
    train_eval_ds = SERWaveformDataset(df_train[~df_train['sumber'].str.endswith('_aug')], mode='eval')
    train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)

    noise_candidates = deteksi_label_noise(train_eval_loader)
    print(f'\n[CHECK] Kandidat label noise terdeteksi: {len(noise_candidates)} file')
    if len(noise_candidates):
        print('\nBreakdown per sumber:')
        print(noise_candidates['sumber'].value_counts())
        print('\nContoh 10 kandidat dengan confidence terendah:')
        display(noise_candidates.sort_values('confidence').head(10))
        noise_candidates.to_csv(os.path.join(LOG_DIR,'label_noise_candidates.csv'), index=False)
else:
    noise_candidates = pd.DataFrame()
    print('Deteksi noise dinonaktifkan.')

## 12. Stage 2 - Fine-tune dengan Noise Downweighting

File yang terdeteksi sebagai kandidat noise diturunkan bobotnya (bukan dihapus - masih ada sinyal valid di dalamnya), lalu training dilanjutkan beberapa epoch dengan LR kecil untuk fine-tune akhir.

**[12.1]** Memotong bobot *loss penalty* (Downweighting) dari sampel berisik (*noisy*) yang baru dideteksi agar tidak menyesatkan arah gradien di siklus Stage 2.


In [ ]:
if DO_NOISE_DETECTION and len(noise_candidates) > 0:
    noisy_paths = set(noise_candidates['path'].values)
    df_train['bobot_noise_adj'] = df_train.apply(
        lambda r: r['bobot'] * NOISE_DOWNWEIGHT if r['path'] in noisy_paths else r['bobot'], axis=1)

    print(f'[ADJUST] {len(noisy_paths)} file di-downweight (bobot × {NOISE_DOWNWEIGHT})')

    # Rebuild sampler dengan bobot baru
    source_w2 = df_train['bobot_noise_adj'].values.astype(np.float32)
    sample_w2 = class_weights[y_train_np] * np.sqrt(np.clip(source_w2, 0.05, 3.0))
    lo2, hi2  = np.percentile(sample_w2, [2, 98])
    sample_w2 = np.clip(sample_w2, lo2, hi2)
    sampler2  = WeightedRandomSampler(weights=torch.DoubleTensor(sample_w2),
                    num_samples=len(sample_w2), replacement=True)
    train_loader2 = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler2,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn, drop_last=False)

    # Fine-tune singkat dengan LR kecil
    STAGE2_EPOCHS = 6
    STAGE2_LR_BACKBONE = BACKBONE_LR * 0.3
    STAGE2_LR_HEAD = HEAD_LR * 0.3

    unfreeze_last_layers(model, last_n=UNFREEZE_LAST_N)
    optimizer2 = torch.optim.AdamW([
        {'params': model.backbone.parameters(),   'lr': STAGE2_LR_BACKBONE},
        {'params': model.pooling.parameters(),    'lr': STAGE2_LR_HEAD},
        {'params': model.classifier.parameters(), 'lr': STAGE2_LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)
    steps2 = math.ceil(len(train_loader2)/GRAD_ACCUM_STEPS) * STAGE2_EPOCHS
    scheduler2 = get_cosine_schedule_with_warmup(optimizer2, num_warmup_steps=int(0.1*steps2), num_training_steps=steps2)

    best_acc2, best_epoch2, no_imp2 = best_acc, 0, 0

    print(f'\n[STAGE2] Stage 2: fine-tune {STAGE2_EPOCHS} epoch dengan noise downweighting')
    print(f'{"Epoch":>5} | {"Loss T":>8} | {"Acc T":>7} | {"Loss V":>8} | {"Acc V":>7} | {"Gap":>6}')
    print('─'*60)

    for epoch in range(1, STAGE2_EPOCHS+1):
        model.train()
        total_loss, total_acc, total_n = 0.0, 0.0, 0
        optimizer2.zero_grad(set_to_none=True)
        for step, (inputs, labels, metas) in enumerate(tqdm(train_loader2, desc=f'Stage2 Ep{epoch}', leave=False), start=1):
            inputs = to_device_inputs(inputs); labels = labels.to(device, non_blocking=True)
            with autocast_ctx():
                logits, _ = model(**inputs)
                loss = criterion(logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader2):
                scaler.unscale_(optimizer2)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer2); scaler.update()
                scheduler2.step()
                optimizer2.zero_grad(set_to_none=True)
            bs = labels.size(0)
            total_loss += loss.item()*GRAD_ACCUM_STEPS*bs; total_acc += akurasi(logits.detach(),labels)*bs; total_n += bs
        train_loss2, train_acc2 = total_loss/total_n, total_acc/total_n
        val_out2 = eval_epoch(val_loader, desc=f'Stage2 Ep{epoch} val')
        val_loss2, val_acc2 = val_out2['loss'], val_out2['acc']
        gap2 = train_acc2 - val_acc2
        flag2 = ''
        if val_acc2 > best_acc2 + MIN_DELTA:
            best_acc2, best_epoch2, no_imp2 = float(val_acc2), int(epoch), 0
            flag2 = ' [BEST]'
            torch.save({'epoch':epoch,'model_state':_get_core(model).state_dict(),
                'model_source':MODEL_SOURCE,'val_acc':best_acc2,'val_loss':float(val_loss2),
                'train_acc':float(train_acc2),'train_loss':float(train_loss2),
                'emosi_list':EMOSI_LIST,'label2idx':LABEL2IDX,'idx2label':IDX2LABEL,
                'class_weights':class_weights.tolist()}, BEST_MODEL_PATH_STG2)
        else:
            no_imp2 += 1
        print(f'{epoch:>5} | {train_loss2:>8.4f} | {train_acc2:>7.4f} | {val_loss2:>8.4f} | {val_acc2:>7.4f} | {gap2:>6.3f}{flag2}')
        if no_imp2 >= 3:
            print(f'[STOP] Stage 2 early stop epoch {epoch}')
            break

    if os.path.exists(BEST_MODEL_PATH_STG2):
        ckpt2 = torch.load(BEST_MODEL_PATH_STG2, map_location=device)
        if ckpt2['val_acc'] > ckpt['val_acc']:
            model.load_state_dict(ckpt2['model_state']); model.eval()
            print(f'\n[OK] Stage 2 lebih baik - pakai checkpoint Stage 2 (val acc: {ckpt2["val_acc"]*100:.2f}%)')
            ckpt = ckpt2
            BEST_MODEL_PATH = BEST_MODEL_PATH_STG2
        else:
            print(f'\n[NOTE] Stage 1 tetap lebih baik (val acc: {ckpt["val_acc"]*100:.2f}% vs Stage 2: {ckpt2["val_acc"]*100:.2f}%)')
else:
    print('Stage 2 dilewati (noise detection nonaktif atau tidak ada kandidat).')

**[12.2]** Memuat kembali *state dictionary* model dari checkpoint terbaik di Stage 1 dan mencetak diagnosis metrik diagnostik perbedaannya.


In [ ]:
ckpt_d = torch.load(BEST_MODEL_PATH, map_location='cpu')
gap_acc  = float(ckpt_d['train_acc']) - float(ckpt_d['val_acc'])
print('='*70)
print('  [STATS] DIAGNOSIS FINAL - WAVLM v7 BEST CHECKPOINT')
print('='*70)
print(f'  Best epoch : {ckpt_d["epoch"]}')
print(f'  Train acc  : {ckpt_d["train_acc"]*100:.2f}%   Val acc : {ckpt_d["val_acc"]*100:.2f}%')
print(f'  Gap acc    : {gap_acc*100:.2f}%')
print('─'*70)
if gap_acc > 0.20: print('  [WARNING]  Masih overfitting - pertimbangkan dropout lebih tinggi.')
elif gap_acc > 0.10: print('  [INFO] Overfitting ringan - normal untuk fine-tuning multi-sumber.')
else: print('  [OK] Generalisasi baik.')
print('='*70)

**[12.3]** Otomatisasi MLOps: Mengunggah model WavLM hasil *finetuning* terbaik beserta eksekusi dari `/kaggle/working` langsung ke *cloud* Hugging Face Hub.


In [ ]:
### [MLOps] Hugging Face Hub Auto-Upload
# Cell ini dikonfigurasi khusus untuk Kaggle Environment menggunakan Kaggle Secrets
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    print("[WARNING] Gagal membaca Kaggle Secrets. Pastikan kamu sudah menambahkan HF_TOKEN di menu Add-ons -> Secrets.")
    HF_TOKEN = None

REPO_ID = 'elnathzzz/IndoWaveSentiment'
MODEL_PATH = '/kaggle/working/ser_wavlm_v7_best.pt'

if os.path.exists(MODEL_PATH) and HF_TOKEN:
    print('[INFO] Melakukan otentikasi ke Hugging Face...')
    login(token=HF_TOKEN)
    
    api = HfApi()
    print(f'[INFO] Mengunggah {MODEL_PATH} ke repositori {REPO_ID}...')
    try:
        api.upload_file(
            path_or_fileobj=MODEL_PATH,
            path_in_repo='ser_wavlm_v7_best.pt',
            repo_id=REPO_ID,
            repo_type='model',
            commit_message='[BEST] Upload WavLM v7 SER model checkpoint from Kaggle'
        )
        print('[OK] Unggahan berhasil! Model sudah berada di Cloud Hugging Face.')
    except Exception as e:
        print(f'[ERROR] Gagal mengunggah: {e}')
elif not HF_TOKEN:
    print('[ERROR] Token HF_TOKEN tidak ditemukan.')
else:
    print(f'[ERROR] File checkpoint {MODEL_PATH} tidak ditemukan. Pastikan proses training selesai.')


## 13. Evaluasi & Visualisasi

**[13.1]** Merender plot *line chart* pergerakan fluktuasi metrik *Loss* dan *Accuracy* (Training vs Validation) antar epoch untuk mendiagnosis konvergensi.


In [ ]:
hist = riwayat
ep_r = range(1, len(hist['la'])+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(ep_r, hist['la'], label='Latih', lw=2)
ax1.plot(ep_r, hist['va'], label='Validasi', lw=2, ls='--')
ax1.set_title('Akurasi per Epoch (Stage 1)'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep_r, hist['ll'], label='Latih', lw=2)
ax2.plot(ep_r, hist['vl'], label='Validasi', lw=2, ls='--')
ax2.set_title('Loss per Epoch (Stage 1)'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Kurva Training v7 - Base-Plus Optimized + Noise Handling', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'kurva_training_v7.png'), dpi=150, bbox_inches='tight')
plt.show()

**[13.2]** Eksekusi inferensi final pada data uji (*Holdout Test Set*) yang tak pernah dilihat model, mencetak *Classification Report* skikit-learn.


In [ ]:
model.load_state_dict(ckpt['model_state']); model.eval()
test_out = eval_epoch(test_loader, desc='test')
pred_all, true_all = test_out['pred'], test_out['true']
test_acc = accuracy_score(true_all, pred_all)

print('='*68)
print('  Laporan Klasifikasi - Data Uji (WavLM v7)')
print('='*68)
print(classification_report(true_all, pred_all, target_names=EMOSI_LIST, digits=4, zero_division=0))
print(f'  Akurasi Keseluruhan : {test_acc*100:.2f}%')
print(f'  Val acc terbaik     : {ckpt["val_acc"]*100:.2f}%')
print('='*68)

**[13.3]** Menghasilkan visualisasi persentase distribusi prediksi silang (Confusion Matrix) dengan balutan Heatmap Seaborn untuk menganalisis akurasi per kelas.


In [ ]:
cm = confusion_matrix(true_all, pred_all, labels=np.arange(NUM_CLASSES))
cm_pct = np.divide(cm.astype(float), cm.sum(axis=1,keepdims=True),
                   out=np.zeros_like(cm,dtype=float), where=cm.sum(axis=1,keepdims=True)!=0)
labels_cm = [f'{EMOJI_MAP[e]} {e}' for e in EMOSI_LIST]
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Blues', xticklabels=labels_cm, yticklabels=labels_cm,
            ax=ax, linewidths=0.5, linecolor='white')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j+0.5, i+0.72, f'({cm[i,j]})', ha='center', va='center', fontsize=8, color='gray')
ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
ax.set_title('Confusion Matrix - WavLM v7', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR,'confusion_matrix_v7.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n[CHECK] Akurasi per kelas:')
for i,e in enumerate(EMOSI_LIST): print(f'  {EMOJI_MAP[e]} {e:<8}: {cm_pct[i,i]*100:.2f}%')

**[13.4]** Mengonstruksi ulang *DataFrame* hasil evaluasi terperinci untuk tiap sampel audio demi kepentingan *Error Analysis* yang mendalam di masa depan.


In [ ]:
eval_df = pd.DataFrame({
    'path':test_out['path'],'true':test_out['true'],'pred':test_out['pred'],
    'emosi_true':[IDX2LABEL[int(i)] for i in test_out['true']],
    'emosi_pred':[IDX2LABEL[int(i)] for i in test_out['pred']],
    'sumber':test_out['sumber'],'bahasa':test_out['bahasa'],
})
eval_df['benar'] = eval_df['true'] == eval_df['pred']
eval_df.to_csv(os.path.join(LOG_DIR,'evaluasi_test_v7.csv'), index=False)

print('[STATS] Akurasi per sumber:')
display(eval_df.groupby('sumber')['benar'].agg(['count','mean']).rename(columns={'count':'n','mean':'akurasi'}).sort_values('akurasi',ascending=False))
print('\n[STATS] Akurasi per bahasa:')
display(eval_df.groupby('bahasa')['benar'].agg(['count','mean']).rename(columns={'count':'n','mean':'akurasi'}))
print('\n[STATS] Akurasi CREMA-D per emosi (cek apakah noise handling membantu):')
display(eval_df[eval_df['sumber']=='cremad'].groupby('emosi_true')['benar'].agg(['count','mean']).rename(columns={'count':'n','mean':'akurasi'}))

## 14. Export Package

**[14.1]** Menyalin file checkpoint `.pt`, konfigurasi fitur ekstraktor, dan mapping ID json ke dalam satu *folder* dan membungkusnya jadi paket instalasi `ZIP`.


In [ ]:
DOWNLOAD_DIR = os.path.join(BASE_DIR,'download_package_v7')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
shutil.copy(BEST_MODEL_PATH, os.path.join(DOWNLOAD_DIR,'ser_wavlm_v7_best.pt'))
FE_DIR = os.path.join(DOWNLOAD_DIR,'feature_extractor')
os.makedirs(FE_DIR, exist_ok=True)
feature_extractor.save_pretrained(FE_DIR)
for src in [HISTORY_PATH, SPLIT_PATH,
            os.path.join(LOG_DIR,'kurva_training_v7.png'),
            os.path.join(LOG_DIR,'confusion_matrix_v7.png'),
            os.path.join(LOG_DIR,'evaluasi_test_v7.csv'),
            os.path.join(LOG_DIR,'label_noise_candidates.csv')]:
    if os.path.exists(src): shutil.copy(src, os.path.join(DOWNLOAD_DIR, os.path.basename(src)))

config_export = {
    'version':'v7-base-plus-optimized-noise-aware',
    'model_source':MODEL_SOURCE,'emosi_list':EMOSI_LIST,
    'max_seconds':MAX_SECONDS,'dropout':DROPOUT,'label_smoothing':LABEL_SMOOTHING,
    'mixup_alpha':MIXUP_ALPHA,'mixup_prob':MIXUP_PROB,
    'cremad_bobot_per_kelas':CREMAD_BOBOT_PER_KELAS,
    'noise_candidates_count': int(len(noise_candidates)) if 'noise_candidates' in dir() else 0,
    'best_epoch':int(ckpt['epoch']),'best_val_acc':float(ckpt['val_acc']),'test_acc':float(test_acc),
}
with open(os.path.join(DOWNLOAD_DIR,'config_v7.json'),'w',encoding='utf-8') as f:
    json.dump(config_export, f, ensure_ascii=False, indent=2)

ZIP_PATH = os.path.join(BASE_DIR,'ser_wavlm_v7_package.zip')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as zf:
    for root,_,files in os.walk(DOWNLOAD_DIR):
        for fname in files:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, DOWNLOAD_DIR))
zip_mb = os.path.getsize(ZIP_PATH)/1024/1024
print(f'[OK] Package: {ZIP_PATH} ({zip_mb:.1f} MB)')
if IS_KAGGLE: print('[LOAD] Download dari tab Output Kaggle → ser_wavlm_v7_package.zip')
else: ipy_display(FileLink(ZIP_PATH, result_html_prefix='[DOWNLOAD] Download: '))

## 15. Inference - Prediksi Audio Baru

**[15.1]** Fungsi *wrapper end-to-end* untuk inferensi: Menerima *path* input audio sembarang dan mengembalikan array distribusi logit/probabilitas kelas emosi.


In [ ]:
@torch.no_grad()
def prediksi_emosi(path_audio, model=model):
    y = load_waveform(path_audio)
    y = fix_length(y, mode='eval')
    inputs = feature_extractor([y], sampling_rate=SAMPLE_RATE, padding=True, truncation=True,
        max_length=MAX_SAMPLES, return_attention_mask=True, return_tensors='pt')
    if 'attention_mask' not in inputs:
        inputs['attention_mask'] = torch.ones_like(inputs['input_values'], dtype=torch.long)
    inputs = to_device_inputs(inputs)
    model.eval()
    logits, attn = model(**inputs)
    prob = F.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    idx  = int(prob.argmax())
    return {'emosi':IDX2LABEL[idx],'keyakinan':float(prob[idx]),
            'semua_prob':{IDX2LABEL[i]:float(prob[i]) for i in range(len(prob))}}

def tampilkan_hasil(hasil, path_audio=None):
    emosi, conf = hasil['emosi'], hasil['keyakinan']
    print('─'*56)
    print(f'  Emosi Terdeteksi : {EMOJI_MAP[emosi]}  {emosi.upper()}')
    print(f'  Keyakinan        : {conf*100:.2f}%')
    print('─'*56)
    for e, p in sorted(hasil['semua_prob'].items(), key=lambda x:-x[1]):
        bar = '█'*int(p*24)
        print(f'  {EMOJI_MAP[e]} {e:<8} {bar:<24} {p*100:6.2f}%')
    print('─'*56)
    if path_audio and os.path.exists(path_audio):
        y_plot, sr_plot = librosa.load(path_audio, sr=SAMPLE_RATE, mono=True)
        fig, ax = plt.subplots(figsize=(10,2.4))
        librosa.display.waveshow(y_plot, sr=sr_plot, ax=ax)
        ax.set_title(f'{EMOJI_MAP[emosi]} {emosi.upper()} — {conf*100:.2f}%')
        plt.tight_layout(); plt.show()

print('[OK] Fungsi inference siap')

**[15.2]** Simulasi demonstrasi inferensi: Menarik sampel data acak secara otomatis dan mengeplot kurva diagram batang probabilitas visual (*bar chart*).


In [ ]:
idx_contoh = 0
row = df_test.reset_index(drop=True).iloc[idx_contoh]
print(f'File: {os.path.basename(row["path"])} | Sumber: {row["sumber"]} | Aktual: {row["emosi"]}')
hasil = prediksi_emosi(row['path'])
tampilkan_hasil(hasil, row['path'])

## [OK] Ringkasan v7

| Komponen | v6 (gagal, 70%) | v7 (koreksi) |
|---|---|---|
| Dropout | 0.50 | 0.30 |
| Unfreeze | semua 24 layer (LLRD) | gradual 6 layer |
| Mixup prob | 50% | 25% |
| Mixup alpha | 0.3 | 0.2 |
| SpecAugment prob | 0.5 | 0.3 |
| Label smoothing | 0.15 | 0.10 |
| Audio durasi | 6.0s | 4.0s |
| CREMA-D bobot | flat 0.4 | per-kelas (0.10–0.957 berdasar riset) |
| Label noise | tidak ditangani | deteksi otomatis + Stage 2 fine-tune |

### Catatan jujur soal target 85% test accuracy
Notebook ini menerapkan semua strategi yang masuk akal secara teknis dan ilmiah. Tapi 85% test accuracy untuk SER 6-kelas multi-corpus berada **di luar rentang umum** hasil riset SOTA dunia (biasanya 70-82% untuk setup serupa). Jika hasil aktual mendarat di 78-83%, itu **sudah merupakan pencapaian yang sangat baik** secara akademis - bukan kegagalan.

Untuk laporan, gunakan temuan CHUCKLE (2024) sebagai justifikasi ilmiah kenapa CREMA-D Sad/Fear/Disgust/Happy sulit, dan jelaskan strategi noise handling sebagai kontribusi metodologis yang valid terlepas dari angka akhirnya.

**[15.3]** Mengaktifkan `IPython FileLink` untuk menyediakan tombol interaktif pengunduhan *package* arsip model secara instan pada antarmuka *notebook*.


In [ ]:
from IPython.display import FileLink

# Untuk file zip
FileLink('ser_wavlm_v7_package.zip')